# Reproducing McGuire, Telian, Laboy-Juarez et al. (2016) from CRCNS ssc-4: decoders (Figure 9)

Paper: doi:10.1371/journal.pbio.1002549. Dataset: CRCNS ssc-4 (doi:10.6080/K06D5QXN),
`ssc-4/live/AllRats.mat`, version `unversioned`, pinned by its MD5 sum
a929b803ca0feab5270e807f571d13eb from the data set's `checksums.md5` (the only version in the
bucket; see the header of `reproduce_mcguire_2016.ipynb` for the reasons and the data checks).
The claims file was written from the version of record with the Methods and supporting legends.

The unit classes are recomputed here exactly as in `reproduce_mcguire_2016.ipynb` (temporal
modulation reading a, regression classes reading a with 55 ms Slow), with the same seed.

| claim | here |
|---|---|
| fig9b-stimulus-decoder-fast-medium | tested |
| fig9c-best-units-short-windows | tested |
| fig9b-stimulus-decoder-slow-chance | tested |
| fig9d-single-bin-decoder | tested |
| fig9e-temporal-pattern-decoding | tested |
| fig9f-choice-decoder | tested |
| fig9g-l5b-slow-positive-best-choice | tested |

Decoder details (helper docstring): logistic regression per unit, L2 with C = 1 (claim reading a)
fit by a batched Newton solver checked against scikit-learn below; reading b (C chosen by 5-fold
cross-validation on the 70%) is run with fewer repeats, as it costs about 30 times more. The
number of repeats of each run is printed with it.

In [ ]:
import collections, time, os
os.environ.setdefault("OMP_NUM_THREADS", "1")
import numpy as np, pandas as pd
from scipy import stats
from sklearn.linear_model import LogisticRegression
import ssc4
pd.set_option("display.width", 200)
t_start = time.time()
D = ssc4.Data(ssc4.load_mat())
U = D.units
pmod = ssc4.temporal_pvals(D, "np", seed=1)
R2, P, B = ssc4.regress_all(D)
R2 = np.nan_to_num(R2, nan=0.0); P = np.nan_to_num(P, nan=1.0); B = np.nan_to_num(B)
U["mod"] = pmod < 0.05
U["cls"], U["best"], _, _ = ssc4.unit_classes(R2, P, B, U["mod"].values, "all", False)
U["cls_dep"] = ssc4.deposited_classes(D)

In [ ]:
print("sessions", len(D.A), "units", len(U), "completed trials", int(D.trials.complete.sum()))
print("completed trials by sequence", D.trials[D.trials.complete].stim.value_counts().to_dict())
print("classes (this attempt)", U.cls.value_counts().to_dict())
print("classes (deposited labels)", U.cls_dep.value_counts().to_dict())
FM = U.uid[U.cls.isin(["Fast", "Medium"])].values
SL = U.uid[U.cls.isin(["SlowPos", "SlowNeg"])].values
FMd = U.uid[U.cls_dep.isin(["Fast", "Medium"])].values
SLd = U.uid[U.cls_dep.isin(["SlowPos", "SlowNeg"])].values
print("Fast/Medium units %d, Slow units %d (deposited: %d, %d)" % (len(FM), len(SL), len(FMd), len(SLd)))
ntr = pd.Series({u: len(D.unit_trials(u)[1]) for u in U.uid})
print("completed trials per unit: min %d, median %d, max %d" % (ntr.min(), ntr.median(), ntr.max()))
X, t = ssc4.unit_features(D, FM[0], "10ms")
print("10-ms features of one unit: shape", X.shape, "count range", X.min(), X.max())
withdraw = D.trials[D.trials.complete].withdraw_t
print("withdrawal after onset (ms), completed trials: min %.0f, mean %.0f, max %.0f" % (withdraw.min(), withdraw.mean(), withdraw.max()))
# the batched solver against scikit-learn on one unit
y = (t.stim == "FFF").values.astype(float)
b = ssc4.fit_logistic(X[None], y[None], np.ones((1, len(y))))
sk = LogisticRegression(C=1.0, tol=1e-10, max_iter=10000).fit(X, y)
print("batched Newton vs scikit-learn lbfgs: max coefficient difference %.2e, intercepts %.5f %.5f"
      % (np.abs(b[0, :-1] - sk.coef_[0]).max(), b[0, -1], sk.intercept_[0]))

In [ ]:
ORDER = ["FFF", "FMS", "SMF", "SSS"]
def show(conf, accs, label):
    cm = pd.DataFrame(conf, index=ORDER, columns=ORDER)
    diag = np.diag(conf)
    fs = [ORDER.index("FMS"), ORDER.index("SMF")]
    A = diag[fs].mean()
    sub = conf[np.ix_(fs, fs)]
    Bv = np.mean(np.diag(sub) / sub.sum(1))
    print("%s: %d repeats, overall %.1f%% (SD over repeats %.1f); per sequence %s; FMS/SMF reading A %.1f%%, reading B %.1f%%"
          % (label, len(accs), 100 * accs.mean(), 100 * accs.std(), dict(zip(ORDER, (100 * diag).round(1))), 100 * A, 100 * Bv))
    print(cm.round(3))
    return dict(overall=100 * accs.mean(), **{s: 100 * d for s, d in zip(ORDER, diag)}, A=100 * A, B=100 * Bv)
RES = {}
def run(name, units, kind, how=None, reps=300, cv=False, seed=0):
    t0 = time.time()
    conf, accs, ua = ssc4.stim_decoder(D, units, kind, how, repeats=reps, seed=seed, cv=cv)
    RES[name] = dict(conf=conf, accs=accs, ua=ua, units=units, summary=show(conf, accs, name))
    print("   (%.0f s)" % (time.time() - t0))

### fig9b-stimulus-decoder-fast-medium

> A decoder built from all Fast and Medium units with 10-ms bins identified the sequence with 83%
> accuracy (74% for FMS to 88% for FFF; chance 25%), and identified FMS and SMF with 78% accuracy.

In [ ]:
run("FM 10ms", FM, "10ms", seed=11)
run("FM 10ms, C by CV (reading b)", FM, "10ms", reps=5, cv=True, seed=12)
run("variant: FM 10ms, deposited labels", FMd, "10ms", reps=100, seed=13)

**Result: reproduced.** With C = 1 (reading a, 300 repeats) the Fast/Medium decoder is 84.1% correct overall, 74.9% for FMS and 84.4% for FFF, and 82.1% on FMS and SMF (reading A; 97.6% under reading B), all within tolerance. With C chosen by cross-validation (5 repeats) it is 76.0% overall and 62.2% for FMS. The data set's labels give 87.1% (100 repeats).

### fig9c-best-units-short-windows

> The individual units with the best stimulus prediction in the Fast/Medium decoder were those
> with 5 to 10 ms best integration windows.

In [ ]:
r = RES["FM 10ms"]
ua = pd.DataFrame(dict(acc=100 * r["ua"], best=U.best[r["units"]].values, cls=U.cls[r["units"]].values))
g = ua.groupby("best").acc.agg(["mean", "size"]).round(1)
print(g.T.to_string())
top = ua.nlargest(int(np.ceil(0.1 * len(ua))), "acc")
print("top 10%% (%d units): best windows %s; from 5 or 10 ms: %d" % (len(top), top.best.value_counts().to_dict(), top.best.isin([5, 10]).sum()))
print("group with the highest mean:", g["mean"].idxmax(), "ms; groups with at least 3 units:", g[g["size"] >= 3]["mean"].idxmax(), "ms")
print("chance 25%; units above 25%:", int((ua.acc > 25).sum()), "of", len(ua))

**Result: reproduced.** Units with a 5-ms best window have the highest mean single-unit accuracy (32.7%, 10 ms 31.5%, longer windows 27.2 to 30.9%), and 17 of the 21 top-ranked units have 5 or 10 ms windows.

### fig9b-stimulus-decoder-slow-chance

> A decoder built from all Slow units with 10-ms bins predicted sequence identity near chance (32%,
> not different from chance).

In [ ]:
run("Slow 10ms", SL, "10ms", seed=21)
run("Slow 10ms, C by CV (reading b)", SL, "10ms", reps=5, cv=True, seed=22)
run("variant: Slow 10ms, deposited labels", SLd, "10ms", reps=100, seed=23)
def chance_tests(name, npseudo=100):
    a = RES[name]["accs"]
    pa = min(1.0, 2 * min(np.mean(a <= 0.25), np.mean(a >= 0.25)))
    n = len(a) * 4 * npseudo
    k = int(round(a.mean() * n))
    pb = stats.binomtest(k, n, 0.25).pvalue
    print("%s: accuracy %.1f%%; (a) two-sided fraction of repeats on the other side of 25%%: p = %.3f; (b) binomial test of %d pooled pseudo-trials: p = %.2g"
          % (name, 100 * a.mean(), pa, n, pb))
for nm in ["Slow 10ms", "Slow 10ms, C by CV (reading b)", "variant: Slow 10ms, deposited labels"]:
    chance_tests(nm)

**Result: reproduced.** The Slow decoder is 30.1% correct (C = 1, 300 repeats), within tolerance of 32%, and not different from chance by the fraction of repeats (p = 0.213, reading a). The binomial test on 120,000 pooled pseudo-trials (reading b) is significant (p near 0), as any small excess over 25% is with that many draws. With C by cross-validation it is 28.9%, significant by both tests.

### fig9d-single-bin-decoder

> Decoding from mean firing rate in a single 150-ms bin reduced Fast/Medium decoder accuracy to 43%
> and did not improve Slow decoder accuracy.

In [ ]:
run("FM 1bin", FM, "1bin", seed=31)
run("Slow 1bin", SL, "1bin", seed=32)
run("FM 1bin, C by CV (reading b)", FM, "1bin", reps=5, cv=True, seed=33)
run("Slow 1bin, C by CV (reading b)", SL, "1bin", reps=5, cv=True, seed=34)
for a, b in [("Slow 1bin", "Slow 10ms"), ("Slow 1bin, C by CV (reading b)", "Slow 10ms, C by CV (reading b)")]:
    d = RES[a]["summary"]["overall"] - RES[b]["summary"]["overall"]
    print("%s minus %s: %.1f points (limit 3.2)" % (a, b, d))

**Result: reproduced.** With one 150-ms bin the Fast/Medium decoder falls to 39.4% (within tolerance of 43%) and the Slow decoder to 26.5%, 3.6 points below its 10-ms value, not above it. With C by cross-validation the values are 28.9% and 24.9%.

### fig9e-temporal-pattern-decoding

> Fast/Medium decoders trained on rate-normalized spike trains performed well (80%), while
> time-scrambling spikes within trials abolished performance.

In [ ]:
run("FM 10ms rate-normalized", FM, "10ms", "norm", seed=41)
run("FM 10ms time-scrambled", FM, "10ms", "scramble", seed=42)
run("FM 10ms rate-normalized, C by CV (reading b)", FM, "10ms", "norm", reps=5, cv=True, seed=43)
run("FM 10ms time-scrambled, C by CV (reading b)", FM, "10ms", "scramble", reps=5, cv=True, seed=44)
for nm in ["FM 10ms time-scrambled", "FM 10ms time-scrambled, C by CV (reading b)"]:
    chance_tests(nm)
    print("   below 32%%: %s" % (RES[nm]["summary"]["overall"] < 32))

**Result: reproduced.** Rate-normalized trains give 79.4% (within tolerance of 80%), time-scrambled trains 28.9%, not above chance by the fraction of repeats (p = 0.367) and below 32%. With C by cross-validation, 72.1% and 28.3%.

### fig9f-choice-decoder

> A choice decoder from Slow units predicted drink port choice at 65% (chance 50%) from a single
> 150-ms bin or from the last 100 ms before nose poke withdrawal, while one from Fast and Medium
> units could not predict choice.

Training weights each choice equally (an assumption: the Methods sum the probabilities and give
chance as 50%, which an unweighted fit on a side-biased rat does not give); the unweighted fit is
shown as a variant.

In [ ]:
CH = {}
def runc(name, units, kind, reps=300, seed=0, balanced=True, cv=False):
    t0 = time.time()
    res, ua = ssc4.choice_decoder(D, units, kind, repeats=reps, seed=seed, balanced=balanced, cv=cv)
    tab = pd.DataFrame(res, index=["i equal draws", "ii data proportions"]).T.loc[ORDER] * 100
    a_i, a_ii = tab.iloc[:, 0].mean(), tab.iloc[:, 1].mean()
    allt = D.trials[D.trials.complete & D.trials.session.isin(U.session[units].unique())]
    w = allt.groupby("stim").size()
    p_i = (tab.iloc[:, 0] * w.loc[ORDER]).sum() / w.sum()
    p_ii = (tab.iloc[:, 1] * w.loc[ORDER]).sum() / w.sum()
    CH[name] = dict(tab=tab, ua=ua, units=units, a_i=a_i, a_ii=a_ii, pooled_i=p_i, pooled_ii=p_ii)
    print("%s: %d repeats; mean over sequences (a): draws i %.1f%%, ii %.1f%%; pooled over trials (b): i %.1f%%, ii %.1f%%  (%.0f s)"
          % (name, reps, a_i, a_ii, p_i, p_ii, time.time() - t0))
    print(tab.round(1).T.to_string())
runc("Slow 1bin", SL, "1bin", seed=51)
runc("Slow pre100", SL, "pre100", seed=52)
runc("FM 1bin", FM, "1bin", seed=53)
runc("FM pre100", FM, "pre100", seed=54)
runc("Slow 1bin, C by CV (reading b)", SL, "1bin", reps=10, seed=55, cv=True)
runc("Slow pre100, C by CV (reading b)", SL, "pre100", reps=10, seed=56, cv=True)
runc("FM 1bin, C by CV (reading b)", FM, "1bin", reps=10, seed=57, cv=True)
runc("FM pre100, C by CV (reading b)", FM, "pre100", reps=10, seed=58, cv=True)
runc("variant: Slow pre100, unweighted training", SL, "pre100", reps=50, seed=59, balanced=False)
runc("variant: FM pre100, unweighted training", FM, "pre100", reps=50, seed=60, balanced=False)
runc("variant: Slow pre100, deposited labels", SLd, "pre100", reps=100, seed=61)
runc("variant: FM pre100, deposited labels", FMd, "pre100", reps=100, seed=62)
print("\nchance: 50% under draws i; under draws ii, the majority choice of each sequence gives",
      {s: round(100 * max(p, 1 - p), 1) for s, p in D.trials[D.trials.complete].groupby("stim").chosen.mean().items()})

**Result: not reproduced.** The Slow choice decoder predicts choice far better than 65%: 80.1% from the 150-ms bin and 91.7% from the 100 ms before withdrawal (equal draws, mean over sequences; 74.6% and 79.2% with C by cross-validation), and the Fast/Medium decoder is not unable (67.1% and 79.0%). Trained without balancing the two choices, both decoders fall to 50 to 54%. None of the readings gives Slow at 65% with Fast/Medium near 50%.

### fig9g-l5b-slow-positive-best-choice

> The units with the best choice prediction were Slow Positive units located primarily in L5b.

Each Slow unit's own classifier on its test trials, one 100-ms bin before withdrawal, scored as
balanced accuracy (mean of right and left hit rates) so that chance is 50%, a standard choice.

In [ ]:
r = CH["Slow pre100"]
ua = pd.DataFrame(dict(acc=100 * r["ua"], layer=U.layer[r["units"]].values, cls=U.cls[r["units"]].values))
g = ua.groupby(["layer", "cls"]).acc.agg(["mean", "size"]).round(1)
print(g.to_string())
print("group with the highest mean:", g["mean"].idxmax(), "| among groups with at least 3 units:", g[g["size"] >= 3]["mean"].idxmax())
top = ua.nlargest(int(np.ceil(0.1 * len(ua))), "acc")
print("top 10%% (%d units): %s; L5b Slow Positive: %d" % (len(top), collections.Counter(zip(top.layer, top.cls)).most_common(),
      int(((top.layer == "L5b") & (top.cls == "SlowPos")).sum())))
print("total run time of this notebook so far: %.1f min" % ((time.time() - t_start) / 60))

**Result: reproduced.** L5b Slow Positive units have the highest mean single-unit choice accuracy (59.8%, 15 units; next L5a Slow Positive 56.4%, L5b Slow Negative 55.4%), and 5 of the 11 top-ranked units are L5b Slow Positive, the largest group but not more than half.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| claim | outcome |
|---|---|
| fig9b-stimulus-decoder-fast-medium | reproduced |
| fig9c-best-units-short-windows | reproduced |
| fig9b-stimulus-decoder-slow-chance | reproduced |
| fig9d-single-bin-decoder | reproduced |
| fig9e-temporal-pattern-decoding | reproduced |
| fig9f-choice-decoder | not reproduced |
| fig9g-l5b-slow-positive-best-choice | reproduced |

How the analysis differs from the paper's: the units are this attempt's classes (208
Fast/Medium, 102 Slow) from all 506 deposited units; the population decoder draws 100
pseudo-trials per class per repeat; the cross-validated penalty (reading b) is run with 5 to 10
repeats; the choice decoder weights the two choices equally in training (an assumption), as the
unweighted fit predicts one side for every trial.